In [ ]:
using ITensors
using Printf
using Random






Random.seed!(1234)


N = 130
J1 = 1
J2=0.253
J3= 0.031
BD=180

print("J2=")
println(J2)
print("J3=")
println(J3)


# Create N spin-one degrees of freedom
#sites = siteinds("S=1", N)
# Alternatively can make spin-half sites instead
sites = siteinds("S=3/2", N)

# Input operator terms which define a Hamiltonian
ampo = AutoMPO()

# Add J1 nearest-neighbor interaction term
 # Adjust the coupling constant as needed
for i in 1:N-1
    for op in ["Sx", "Sy", "Sz"]
        add!(ampo, J1, op, i, op, i+1)
    end
end



for i in 1:N-2
    for op in ["Sx", "Sy", "Sz"]
        add!(ampo, J2, op, i, op, i+2)
    end
end




for i in 2:N-1
    for op1 in ["Sx", "Sy", "Sz"]
        for op2 in ["Sx", "Sy", "Sz"]
            add!(ampo, J3, op1, i-1, op1, i, op2, i, op2, i+1)
            add!(ampo, J3, op1, i, op1, i+1, op2, i-1, op2, i)
        end
    end
end

H = MPO(ampo, sites)

# Create an initial random matrix product state
psi0 = randomMPS(sites; linkdims=20)

# Plan to do 5 DMRG sweeps:
nsweeps = 20
# Set maximum MPS bond dimensions for each sweep
maxdim = [BD]
# Set maximum truncation error allowed when adapting bond dimensions
cutoff = [1E-12]

# Run the DMRG algorithm, returning energy and optimized MPS
energy, psi = dmrg(H, psi0; nsweeps, maxdim,cutoff=cutoff)
@printf("Final energy = %.12f\n", energy/N)


orthogonalize!(psi,1)

using ITensors.HDF5
f = h5open("j1j2j3spin3by2,L="*string(N)*",BD="*string(maxdim[1])*",J2="*string(J2)*"J3="*string(J3)*".h5","w")
write(f,"psi",psi)
close(f)


using ITensors.HDF5
f = h5open("j1j2j3spin3by2,L="*string(N)*",J2="*string(J2)*",J3="*string(J3)*".h5","w")
write(f,"mpo",H)
close(f)


a=inner(psi',H,psi)
psin=apply(H,psi)
b=inner(psin',psin)
error=b-a^2
println(error)


H = nothing

psi= nothing

# Call garbage collection to free up memory
GC.gc()